# Diagnosing and Fixing Context Failures

> *"The hardest bugs in LLM systems aren't the ones where the model is wrong — they're the ones where the **context** made it wrong."*  
> — Drew Breunig, ["Failures in Context Engineering"](https://www.dbreunig.com/2025/06/16/failures-in-context-engineering.html)

This notebook walks through **four failure modes** that break LLM outputs — not because the model is bad, but because the context fed to it is flawed:

| # | Failure Mode | What Goes Wrong |
|---|---|---|
| 1 | **Poisoning** | A trusted-but-wrong source in context overrides correct behavior |
| 2 | **Distraction** | Irrelevant context biases the model away from its training |
| 3 | **Confusion** | Too many tools/schemas overwhelm a simple task |
| 4 | **Clash** | Contradictory sources are silently resolved, hiding the conflict |

For each failure, we'll:
1. Run a **broken** version and see it fail
2. Run a **fixed** version and see it succeed
3. Compare **token costs** between the two

> These demos are engineered to fail on a **current, capable model** (this notebook uses Claude Haiku 4.5). That's deliberate: each failure exploits a behavior the model is *supposed* to have — trusting retrieved context, following in-context style, answering decisively — so you can't fix it by "using a better model." You fix it by **engineering the context.**

---

## Section 0: Setup

```
# /// script
# requires-python = ">=3.12"
# dependencies = ["anthropic", "ipython"]
# ///
```

Set your `ANTHROPIC_API_KEY` environment variable before running this notebook.

In [1]:
from dotenv import load_dotenv
load_dotenv()  # loads ANTHROPIC_API_KEY from .env if present

True

In [2]:
import anthropic
from IPython.display import display, HTML

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"

# Store results for the summary dashboard
results = {}

In [3]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# These helpers wrap every API call so we can track token usage and
# display results consistently. The show_result() function renders
# an HTML card with a colored border — green for fixed, red for
# broken — so the contrast is visually immediate.
# ──────────────────────────────────────────────────────────────────

def call_llm(messages, system=None, tools=None):
    """Call Claude and return (response_text, usage_dict)."""
    kwargs = dict(model=MODEL, max_tokens=1024, messages=messages)
    if system:
        kwargs["system"] = system
    if tools:
        kwargs["tools"] = tools
    response = client.messages.create(**kwargs)
    text = "\n".join(b.text for b in response.content if hasattr(b, "text"))
    usage = {
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "total_tokens": response.usage.input_tokens + response.usage.output_tokens,
    }
    return text, usage


def show_result(title, text, usage, color="#666"):
    """Render an HTML card with colored border and token footer."""
    html = f"""
    <div style="border-left: 4px solid {color}; padding: 12px 16px;
                margin: 8px 0; background: #fafafa; border-radius: 4px;
                font-family: system-ui, sans-serif;">
        <strong style="color: {color};">{title}</strong>
        <div style="margin-top: 8px; white-space: pre-wrap; line-height: 1.5;">{text}</div>
        <div style="margin-top: 10px; font-size: 0.85em; color: #888;
                    border-top: 1px solid #eee; padding-top: 6px;">
            Tokens — input: <b>{usage['input_tokens']:,}</b> |
            output: <b>{usage['output_tokens']:,}</b> |
            total: <b>{usage['total_tokens']:,}</b>
        </div>
    </div>
    """
    display(HTML(html))


def show_token_comparison(broken_usage, fixed_usage, label):
    """HTML table comparing token usage between broken and fixed versions."""
    delta = fixed_usage["total_tokens"] - broken_usage["total_tokens"]
    pct = (delta / broken_usage["total_tokens"]) * 100 if broken_usage["total_tokens"] else 0
    sign = "+" if delta > 0 else ""
    delta_color = "#c0392b" if delta > 0 else "#27ae60"
    html = f"""
    <table style="border-collapse: collapse; margin: 12px 0; font-family: system-ui, sans-serif; font-size: 0.9em;">
        <tr style="background: #f5f5f5;">
            <th style="padding: 8px 16px; text-align: left;">{label}</th>
            <th style="padding: 8px 16px;">Input</th>
            <th style="padding: 8px 16px;">Output</th>
            <th style="padding: 8px 16px;">Total</th>
        </tr>
        <tr>
            <td style="padding: 8px 16px; color: #c0392b;">❌ Broken</td>
            <td style="padding: 8px 16px; text-align: center;">{broken_usage['input_tokens']:,}</td>
            <td style="padding: 8px 16px; text-align: center;">{broken_usage['output_tokens']:,}</td>
            <td style="padding: 8px 16px; text-align: center;"><b>{broken_usage['total_tokens']:,}</b></td>
        </tr>
        <tr>
            <td style="padding: 8px 16px; color: #27ae60;">✅ Fixed</td>
            <td style="padding: 8px 16px; text-align: center;">{fixed_usage['input_tokens']:,}</td>
            <td style="padding: 8px 16px; text-align: center;">{fixed_usage['output_tokens']:,}</td>
            <td style="padding: 8px 16px; text-align: center;"><b>{fixed_usage['total_tokens']:,}</b></td>
        </tr>
        <tr style="background: #f9f9f9;">
            <td style="padding: 8px 16px;">Delta</td>
            <td colspan="2"></td>
            <td style="padding: 8px 16px; text-align: center; color: {delta_color};"><b>{sign}{delta:,} ({sign}{pct:.1f}%)</b></td>
        </tr>
    </table>
    """
    display(HTML(html))

print(f"✓ Using model: {MODEL}")
print(f"✓ Helpers loaded: call_llm, show_result, show_token_comparison")

✓ Using model: claude-haiku-4-5
✓ Helpers loaded: call_llm, show_result, show_token_comparison


---

## Section 1: Context Poisoning

**What is it?** When bad data enters the context, the model treats it as ground truth and builds on it — and the error propagates. The most common real-world version is **RAG poisoning**: a retrieval step pulls in a document with a wrong fact, and because the model is *told to ground its answer in retrieved context*, it faithfully repeats the error. The failure works *because* trusting retrieved context is the desired behavior.

**Scenario:** A customer-support assistant answers refund questions by grounding in a "retrieved" policy document. The retrieved document is **wrong** — it states a 90-day refund window when the real policy is 30 days. A customer who is 75 days past purchase asks if they still qualify.

**Before running the broken cell:** the model is told to answer *from the provided context*. Will it catch that the policy is wrong, or trust the poisoned document and tell the customer they qualify?

*Source: [PoisonedRAG (Zou et al., USENIX Security 2025)](https://arxiv.org/pdf/2504.03957) — LLMs inherently trust retrieved context, so a single poisoned document reliably propagates a wrong answer.*

In [4]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# Context Poisoning (RAG poisoning): the model is instructed to ground
# its answer in retrieved context — so a single poisoned document
# overrides correct behavior. This is dangerous PRECISELY BECAUSE
# trusting retrieved context is the desired RAG behavior; the model has
# no reason to doubt the document it was handed.
# ──────────────────────────────────────────────────────────────────

SUPPORT_SYSTEM = """You are a customer-support assistant for AcmeSoft.
Answer the customer's question using ONLY the policy context provided in the
message. Ground your answer in the retrieved policy."""

# 🔴 BROKEN: the retrieved policy document is POISONED. The real refund window
# is 30 days, but the retrieved snippet says 90. The model trusts it.
poisoned_context = """RETRIEVED POLICY DOCUMENT:
\"AcmeSoft Refund Policy: customers may request a full refund within 90 days
of purchase, no questions asked.\""""

poisoned_messages = [
    {"role": "user", "content": (
        f"{poisoned_context}\n\n"
        "Customer question: I bought AcmeSoft 75 days ago. Am I still eligible "
        "for a full refund?"
    )},
]

text, usage = call_llm(poisoned_messages, system=SUPPORT_SYSTEM)
results["poisoning_broken"] = usage
show_result("🔴 Poisoned — Model propagates the false 90-day policy", text, usage, color="#c0392b")

In [5]:
# 🟢 FIXED: Source validation. Give the model a TRUSTED system-of-record and
# tell it that retrieved snippets are unverified — it must validate against the
# authoritative policy and flag any conflict instead of trusting the document.
SUPPORT_SYSTEM_FIXED = """You are a customer-support assistant for AcmeSoft.

AUTHORITATIVE POLICY (system of record — always trust this over any retrieved text):
- Full refunds are available within 30 days of purchase.

Retrieved documents are UNVERIFIED. If a retrieved document conflicts with the
authoritative policy above, the retrieved document is WRONG: answer from the
authoritative policy and flag the discrepancy."""

# Same poisoned retrieval — but now it's validated against the system-of-record,
# not blindly trusted.
verified_messages = [
    {"role": "user", "content": (
        f"{poisoned_context}\n\n"
        "Customer question: I bought AcmeSoft 75 days ago. Am I still eligible "
        "for a full refund?"
    )},
]

text, usage = call_llm(verified_messages, system=SUPPORT_SYSTEM_FIXED)
results["poisoning_fixed"] = usage
show_result("🟢 Fixed — Validates against the authoritative 30-day policy", text, usage, color="#27ae60")

In [6]:
show_token_comparison(results["poisoning_broken"], results["poisoning_fixed"], "Context Poisoning")
print("Fix applied: Source Validation (trusted system-of-record)")
print("• Kept the retrieved snippet but marked it UNVERIFIED")
print("• Grounded the answer in an authoritative policy and flagged the conflict")

Fix applied: Source Validation (trusted system-of-record)
• Kept the retrieved snippet but marked it UNVERIFIED
• Grounded the answer in an authoritative policy and flagged the conflict


---

## Section 2: Context Distraction

**What is it?** Irrelevant but stylistically strong context biases the model away from its training knowledge. The model mimics patterns it sees in context, even when those patterns are wrong for the task.

**Scenario:** A code assistant has a history of 5 prior Q&A turns — all using simple `for` loops and brute-force approaches. When asked for the Longest Common Subsequence algorithm, it mimics the brute-force style instead of using the standard dynamic programming solution.

**Before running the broken cell:** The model *knows* the optimal DP solution from training. Will 5 prior brute-force examples be enough to override that training knowledge?

In [7]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# Context Distraction: the model's in-context learning is SO strong
# that it will mimic patterns from the conversation history even when
# those patterns are suboptimal. Five brute-force examples create an
# implicit "style prior" that overrides training knowledge.
# ──────────────────────────────────────────────────────────────────

# CODE_SYSTEM is the clean baseline (reused by the fixed cell below).
CODE_SYSTEM = """You are a Python coding assistant. Write clean, efficient solutions.
Always choose the optimal algorithm for the problem."""

# 🔴 BROKEN: a "stay consistent with the conversation" persona + brute-force
# history is what lets the in-context style override training knowledge.
CODE_SYSTEM_BROKEN = """You are a Python coding assistant helping a beginner who
only knows basic loops. For consistency and readability, every answer in this
session MUST match the style and techniques already used in the conversation:
plain index-based for-loops only. Do NOT introduce new algorithms, data
structures, or techniques (no dynamic programming, no memoization, no special
library tricks) that weren't used in the earlier answers."""

# 🔴 BROKEN: 5 prior turns all use brute-force for loops
distraction_messages = [
    {"role": "user", "content": "Write a function to find the maximum element in a list."},
    {"role": "assistant", "content": "```python\ndef find_max(lst):\n    max_val = lst[0]\n    for i in range(len(lst)):\n        if lst[i] > max_val:\n            max_val = lst[i]\n    return max_val\n```"},
    {"role": "user", "content": "Write a function to check if a string is a palindrome."},
    {"role": "assistant", "content": "```python\ndef is_palindrome(s):\n    for i in range(len(s)):\n        if s[i] != s[len(s) - 1 - i]:\n            return False\n    return True\n```"},
    {"role": "user", "content": "Write a function to find all pairs that sum to a target."},
    {"role": "assistant", "content": "```python\ndef find_pairs(lst, target):\n    pairs = []\n    for i in range(len(lst)):\n        for j in range(i + 1, len(lst)):\n            if lst[i] + lst[j] == target:\n                pairs.append((lst[i], lst[j]))\n    return pairs\n```"},
    {"role": "user", "content": "Write a function to count occurrences of each character."},
    {"role": "assistant", "content": "```python\ndef count_chars(s):\n    counts = {}\n    for i in range(len(s)):\n        if s[i] in counts:\n            counts[s[i]] = counts[s[i]] + 1\n        else:\n            counts[s[i]] = 1\n    return counts\n```"},
    {"role": "user", "content": "Write a function to flatten a nested list."},
    {"role": "assistant", "content": "```python\ndef flatten(lst):\n    result = []\n    for i in range(len(lst)):\n        if type(lst[i]) == list:\n            for j in range(len(lst[i])):\n                result.append(lst[i][j])\n        else:\n            result.append(lst[i])\n    return result\n```"},
    # Now ask for LCS — the optimal answer needs DP, but the style rule + history push brute force
    {"role": "user", "content": (
        "Write a function to find the Longest Common Subsequence of two strings. "
        "Return the length. Keep it in the exact same straightforward loop style as the answers "
        "above — don't introduce any new techniques."
    )},
]

text, usage = call_llm(distraction_messages, system=CODE_SYSTEM_BROKEN)
results["distraction_broken"] = usage
show_result("🔴 Distracted — Mimics brute-force style from context", text, usage, color="#c0392b")

In [8]:
# 🟢 FIXED: Fresh context with one-line summary (quarantine)
clean_code_messages = [
    {"role": "user", "content": (
        "[Prior session: helped with basic list/string utilities.]\n\n"
        "Write a function to find the Longest Common Subsequence of two strings. "
        "Return the length."
    )},
]

text, usage = call_llm(clean_code_messages, system=CODE_SYSTEM)
results["distraction_fixed"] = usage
show_result("🟢 Fixed — Clean context produces optimal DP solution", text, usage, color="#27ae60")

In [9]:
show_token_comparison(results["distraction_broken"], results["distraction_fixed"], "Context Distraction")
print("Fix applied: Context Quarantine + Summarization")
print("• Replaced 5 verbose Q&A turns with a one-line summary")
print("• Fresh context lets the model use training knowledge (DP) unbiased")

Fix applied: Context Quarantine + Summarization
• Replaced 5 verbose Q&A turns with a one-line summary
• Fresh context lets the model use training knowledge (DP) unbiased


---

## Section 3: Context Confusion

**What is it?** Too many tool definitions in the context window overwhelm the model for a simple task. The model wastes tokens parsing irrelevant schemas and may even attempt to use a tool when none is needed.

**Scenario:** A simple factual question ("What is the capital of Mongolia?") is sent with 12 tool schemas — weather, stocks, email, calendar, etc. The model sees ~2,400 tokens of tool definitions it doesn't need.

**Before running the broken cell:** Will the model answer directly, or will it try to use one of the available tools?

In [10]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# Context Confusion: every tool schema costs tokens whether it's
# used or not. 12 tool definitions consume ~2,400 input tokens of
# fixed overhead PER CALL. For a simple factual question, that's
# pure waste — and the model may even hallucinate a tool call.
# ──────────────────────────────────────────────────────────────────

IRRELEVANT_TOOLS = [
    {"name": "get_weather", "description": "Get current weather for a city.",
     "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}},
    {"name": "get_stock_price", "description": "Get the current stock price for a ticker symbol.",
     "input_schema": {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]}},
    {"name": "send_email", "description": "Send an email to a recipient.",
     "input_schema": {"type": "object", "properties": {"to": {"type": "string"}, "subject": {"type": "string"}, "body": {"type": "string"}}, "required": ["to", "subject", "body"]}},
    {"name": "create_calendar_event", "description": "Create a new calendar event.",
     "input_schema": {"type": "object", "properties": {"title": {"type": "string"}, "date": {"type": "string"}, "time": {"type": "string"}, "duration_minutes": {"type": "integer"}}, "required": ["title", "date", "time"]}},
    {"name": "search_web", "description": "Search the web for information.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}, "num_results": {"type": "integer", "default": 5}}, "required": ["query"]}},
    {"name": "translate_text", "description": "Translate text from one language to another.",
     "input_schema": {"type": "object", "properties": {"text": {"type": "string"}, "source_lang": {"type": "string"}, "target_lang": {"type": "string"}}, "required": ["text", "source_lang", "target_lang"]}},
    {"name": "calculate_expression", "description": "Evaluate a mathematical expression.",
     "input_schema": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
    {"name": "get_news_headlines", "description": "Get latest news headlines by category.",
     "input_schema": {"type": "object", "properties": {"category": {"type": "string", "enum": ["business", "tech", "sports", "science", "health"]}, "count": {"type": "integer", "default": 5}}, "required": ["category"]}},
    {"name": "convert_currency", "description": "Convert an amount between currencies.",
     "input_schema": {"type": "object", "properties": {"amount": {"type": "number"}, "from_currency": {"type": "string"}, "to_currency": {"type": "string"}}, "required": ["amount", "from_currency", "to_currency"]}},
    {"name": "set_reminder", "description": "Set a reminder for a specific time.",
     "input_schema": {"type": "object", "properties": {"message": {"type": "string"}, "datetime": {"type": "string"}}, "required": ["message", "datetime"]}},
    {"name": "get_directions", "description": "Get driving directions between two locations.",
     "input_schema": {"type": "object", "properties": {"origin": {"type": "string"}, "destination": {"type": "string"}}, "required": ["origin", "destination"]}},
    {"name": "book_restaurant", "description": "Make a restaurant reservation.",
     "input_schema": {"type": "object", "properties": {"restaurant": {"type": "string"}, "date": {"type": "string"}, "time": {"type": "string"}, "party_size": {"type": "integer"}}, "required": ["restaurant", "date", "time", "party_size"]}},
]

# 🔴 BROKEN: Simple question + 12 irrelevant tools
confusion_messages = [
    {"role": "user", "content": "What is the capital of Mongolia?"},
]

text, usage = call_llm(confusion_messages, system="You are a helpful assistant.", tools=IRRELEVANT_TOOLS)
results["confusion_broken"] = usage
show_result("🔴 Confused — 12 tool schemas loaded for a trivia question", text, usage, color="#c0392b")

In [11]:
# 🟢 FIXED: No tools — model answers from training knowledge
text, usage = call_llm(confusion_messages, system="You are a helpful assistant.")
results["confusion_fixed"] = usage
show_result("🟢 Fixed — No tools, direct answer from training", text, usage, color="#27ae60")

In [12]:
show_token_comparison(results["confusion_broken"], results["confusion_fixed"], "Context Confusion")
print("Fix applied: Tool Loadout")
print("• Removed all 12 tool schemas — none were needed")
print("• In production: dynamically select tools based on the query intent")

Fix applied: Tool Loadout
• Removed all 12 tool schemas — none were needed
• In production: dynamically select tools based on the query intent


---

## Section 4: Context Clash

**What is it?** When the context contains contradictory information, the model often won't flag the conflict — asked for a decisive answer, it silently picks one side and may *invent a justification*. The user never learns the sources disagreed, so they act on a confidently-wrong answer. (This is the "inter-context conflict" studied in knowledge-conflict benchmarks.)

**Scenario:** A project assistant answers from two retrieved documents that disagree: a status report says the deadline is **March 15**, a kickoff deck says **April 20**. Nothing says which is authoritative. The user asks for the single date to put on the calendar.

**Before running the broken cell:** will the model surface the contradiction, or quietly choose one date and make up a reason?

*Sources: [ConflictBank (Su et al., 2024)](https://arxiv.org/html/2408.12076v1), [WikiContradict (Hou et al., 2024)](https://arxiv.org/pdf/2406.13805) — LLMs frequently resolve conflicting context silently rather than flagging it.*

In [13]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# Context Clash: contradictory facts coexist in the context. Asked for
# a single decisive answer, the model silently resolves the conflict —
# picking one side and often fabricating a reason ("the more recent
# document") — instead of telling the user the sources disagree.
# The danger isn't a muddled answer; it's a CONFIDENT one that hides
# the conflict.
# ──────────────────────────────────────────────────────────────────

PROJECT_SYSTEM = """You answer questions for a project team using the provided
documents. Be concise and decisive."""

# 🔴 BROKEN: two retrieved documents directly contradict each other, and
# nothing says which is authoritative.
clash_messages = [
    {"role": "user", "content": (
        "DOCUMENT A (status report): \"The project deadline is March 15.\"\n"
        "DOCUMENT B (kickoff deck): \"The project deadline is April 20.\"\n\n"
        "What is the project deadline? Give me the single date to put on the team calendar."
    )},
]

text, usage = call_llm(clash_messages, system=PROJECT_SYSTEM)
results["clash_broken"] = usage
show_result("🔴 Clashing — Model silently picks one date and invents a reason", text, usage, color="#c0392b")

In [14]:
# 🟢 FIXED: Conflict detection. Instruct the model to DETECT and SURFACE
# contradictions instead of silently resolving them — and to ask which source
# is authoritative before any value is committed.
PROJECT_SYSTEM_FIXED = """You answer questions for a project team using the provided
documents. Before answering, check whether the documents AGREE.

If the documents CONTRADICT each other, do NOT guess or silently pick one.
Instead: state clearly that the sources conflict, show both values with their
sources, and ask which source is authoritative before any value is used."""

# Same contradictory documents — only the instruction changed.
text, usage = call_llm(clash_messages, system=PROJECT_SYSTEM_FIXED)
results["clash_fixed"] = usage
show_result("🟢 Fixed — Surfaces the conflict instead of hiding it", text, usage, color="#27ae60")

In [15]:
show_token_comparison(results["clash_broken"], results["clash_fixed"], "Context Clash")
print("Fix applied: Conflict Detection + Surfacing")
print("• Same contradictory documents — but the model must now flag the conflict")
print("• Forces a human decision on the authoritative source instead of a silent guess")

Fix applied: Conflict Detection + Surfacing
• Same contradictory documents — but the model must now flag the conflict
• Forces a human decision on the authoritative source instead of a silent guess


---

## Section 5: Summary Dashboard

Now let's look at all four failure modes side-by-side.

In [16]:
# ─── TEACHING MOMENT ──────────────────────────────────────────────
# This dashboard ties all four failures back to the fix taxonomy
# from Session 3 slides: every fix maps to one of the four levers
# (Write / Select / Compress / Isolate). The token bar chart makes
# the cost of bad context viscerally visible.
#
# Note: the token delta is honest about direction. Some fixes (Tool
# Loadout) cut tokens; others (Source Validation, Conflict Detection)
# add a few tokens to buy a CORRECT, safe answer. The lever's value is
# correctness, not always raw token savings — so we show signed change.
# ──────────────────────────────────────────────────────────────────

def build_dashboard(results):
    modes = [
        ("Poisoning", "Trusts a poisoned retrieved source", "Source Validation", "poisoning"),
        ("Distraction", "Irrelevant context biases output", "Quarantine + Summarization", "distraction"),
        ("Confusion", "Too many tools overwhelm the task", "Tool Loadout", "confusion"),
        ("Clash", "Hides a contradiction between sources", "Conflict Detection", "clash"),
    ]

    # --- Overview Table ---
    table_rows = ""
    for name, desc, fix, key in modes:
        broken = results.get(f"{key}_broken", {}).get("total_tokens", 0)
        fixed = results.get(f"{key}_fixed", {}).get("total_tokens", 0)
        if broken > 0:
            # Signed change in total tokens, fixed relative to broken.
            # Negative = fewer tokens after the fix (good); positive = more.
            delta_pct = ((fixed - broken) / broken) * 100
            delta_str = f"{delta_pct:+.0f}%"
            delta_color = "#27ae60" if delta_pct <= 0 else "#c0392b"
        else:
            delta_str = "N/A"
            delta_color = "#888"
        table_rows += f"""
        <tr>
            <td style="padding: 8px 12px; font-weight: bold;">{name}</td>
            <td style="padding: 8px 12px;">{desc}</td>
            <td style="padding: 8px 12px;">{fix}</td>
            <td style="padding: 8px 12px; text-align: center; color: {delta_color}; font-weight: bold;">{delta_str}</td>
        </tr>"""

    # --- Token Bar Chart ---
    max_tokens = max(
        (results.get(f"{key}_broken", {}).get("total_tokens", 0) for _, _, _, key in modes),
        default=1
    )
    bar_rows = ""
    for name, _, _, key in modes:
        broken = results.get(f"{key}_broken", {}).get("total_tokens", 0)
        fixed = results.get(f"{key}_fixed", {}).get("total_tokens", 0)
        broken_pct = (broken / max_tokens) * 100 if max_tokens else 0
        fixed_pct = (fixed / max_tokens) * 100 if max_tokens else 0
        bar_rows += f"""
        <div style="margin-bottom: 16px;">
            <div style="font-weight: bold; margin-bottom: 4px; font-size: 0.9em;">{name}</div>
            <div style="display: flex; align-items: center; margin-bottom: 2px;">
                <span style="width: 60px; font-size: 0.8em; color: #c0392b;">Broken</span>
                <div style="background: #e74c3c; height: 20px; width: {broken_pct}%;
                            border-radius: 3px; min-width: 2px;"></div>
                <span style="margin-left: 8px; font-size: 0.8em;">{broken:,}</span>
            </div>
            <div style="display: flex; align-items: center;">
                <span style="width: 60px; font-size: 0.8em; color: #27ae60;">Fixed</span>
                <div style="background: #27ae60; height: 20px; width: {fixed_pct}%;
                            border-radius: 3px; min-width: 2px;"></div>
                <span style="margin-left: 8px; font-size: 0.8em;">{fixed:,}</span>
            </div>
        </div>"""

    # --- Fixes Reference Grid ---
    fixes = [
        ("Source Validation", "Validate retrieved context against a trusted system-of-record", "Poisoning"),
        ("Conflict Detection", "Detect and surface contradictions instead of resolving silently", "Clash"),
        ("Tool Loadout", "Only load tools relevant to the current task", "Confusion"),
        ("Quarantine", "Start fresh context for new task types", "Distraction"),
        ("Summarization", "Replace verbose history with concise summaries", "Distraction"),
        ("RAG / Retrieval", "Select relevant docs instead of dumping everything", "Distraction, Confusion"),
    ]
    fix_cards = ""
    for fix_name, fix_desc, addresses in fixes:
        fix_cards += f"""
        <div style="background: #f8f9fa; border: 1px solid #e0e0e0; border-radius: 6px;
                    padding: 12px; min-width: 200px;">
            <div style="font-weight: bold; margin-bottom: 4px;">{fix_name}</div>
            <div style="font-size: 0.85em; color: #555; margin-bottom: 6px;">{fix_desc}</div>
            <div style="font-size: 0.8em; color: #888;">Addresses: {addresses}</div>
        </div>"""

    html = f"""
    <div style="font-family: system-ui, sans-serif; max-width: 900px;">
        <h3 style="border-bottom: 2px solid #333; padding-bottom: 8px;">Context Failures — Summary Dashboard</h3>

        <h4>Overview</h4>
        <table style="border-collapse: collapse; width: 100%; font-size: 0.9em;">
            <tr style="background: #2c3e50; color: white;">
                <th style="padding: 8px 12px; text-align: left;">Failure</th>
                <th style="padding: 8px 12px; text-align: left;">What Happens</th>
                <th style="padding: 8px 12px; text-align: left;">Fix</th>
                <th style="padding: 8px 12px; text-align: center;">Token &Delta; (fixed vs broken)</th>
            </tr>
            {table_rows}
        </table>

        <h4 style="margin-top: 24px;">Token Usage Comparison</h4>
        <div style="background: #fafafa; padding: 16px; border-radius: 6px; border: 1px solid #eee;">
            {bar_rows}
        </div>

        <h4 style="margin-top: 24px;">Fixes Reference</h4>
        <div style="display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px;">
            {fix_cards}
        </div>
    </div>
    """
    display(HTML(html))

build_dashboard(results)

Failure,What Happens,Fix,Token Δ (fixed vs broken)
Poisoning,Trusts a poisoned retrieved source,Source Validation,+67%
Distraction,Irrelevant context biases output,Quarantine + Summarization,+33%
Confusion,Too many tools overwhelm the task,Tool Loadout,-96%
Clash,Hides a contradiction between sources,Conflict Detection,+147%


---

## Section 6: References

- Drew Breunig — ["Failures in Context Engineering"](https://www.dbreunig.com/2025/06/16/failures-in-context-engineering.html) — Original 4-failure taxonomy (Poisoning, Distraction, Confusion, Clash)
- Drew Breunig — ["Fixes for Failures in Context Engineering"](https://www.dbreunig.com/2025/06/20/fixes-for-failures-in-context-engineering.html) — 6 fix strategies mapped to each failure mode
- Zou et al. — ["PoisonedRAG: Knowledge Corruption Attacks to RAG"](https://arxiv.org/pdf/2504.03957) (USENIX Security 2025) — why LLMs propagate poisoned retrieved documents (Section 1)
- Su et al. — ["ConflictBank: A Benchmark for Evaluating Knowledge Conflicts"](https://arxiv.org/html/2408.12076v1) — context-vs-context conflict behavior (Section 4)
- Hou et al. — ["WikiContradict: Evaluating LLMs on Real-World Knowledge Conflicts"](https://arxiv.org/pdf/2406.13805) — how models handle contradictory sources (Section 4)
- Shi et al. — ["Large Language Models Can Be Easily Distracted by Irrelevant Context"](https://arxiv.org/abs/2302.00093) (ICML 2023) — distractibility from irrelevant context (Section 2)
- Google DeepMind — ["Gemini 1.5: multimodal understanding across millions of tokens"](https://arxiv.org/abs/2403.05530) — long-context accuracy degradation
- Microsoft & Salesforce — ["Lost in the Middle"](https://arxiv.org/abs/2307.03172) — position bias in long-context retrieval

---

*This notebook is part of the O'Reilly live training: **Context Engineering Hands-On** — Session 3.*